# Aircraft ETL to Neo4j

This notebook loads the **complete** Aircraft Digital Twin dataset from Databricks Unity Catalog into Neo4j Aura using the Neo4j Spark Connector. It is a guided walkthrough: you start with the core topology (Aircraft, Systems, Components) and then load the rest of the graph (Sensors, Airports, Flights, Delays, Maintenance Events, Removals) that Labs 3 and 4 depend on.

## What You'll Learn
- How to read CSV data from Unity Catalog Volumes
- How to transform tabular data for graph loading
- How to write nodes and relationships using the Neo4j Spark Connector
- How to add constraints so relationships match nodes efficiently
- How to verify the loaded graph with Cypher queries from Databricks

## Prerequisites
- Neo4j Aura credentials from Lab 1
- Access to the workshop Databricks cluster (with Neo4j Spark Connector installed)

## Instructions
1. Clone this notebook to your personal folder
2. Enter your Neo4j credentials in the Configuration cell below
3. Run all cells in order (Shift+Enter or Run All)
4. Verify the results in the final cells

## Section 1: Configuration

Enter your Neo4j Aura connection details below. You received these credentials when you created your Neo4j Aura instance in Lab 1.

Set `CLEAR_DATABASE = True` to wipe the database before loading. Because this notebook loads the full canonical dataset that Labs 3 and 4 build on, clearing first keeps re-runs clean and avoids duplicate data.

In [ ]:
# ============================================
# CONFIGURATION - Enter your Neo4j credentials
# ============================================

NEO4J_URI = ""  # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""  # Your password from Lab 1

# Set to True to clear the database before loading
CLEAR_DATABASE = True

# Unity Catalog Volume path (pre-configured by workshop admin)
DATA_PATH = "/Volumes/databricks-neo4j-workshop/aircraft/raw_data"

# Validate configuration
if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running the notebook!")
else:
    print("Configuration ready!")
    print(f"Neo4j URI: {NEO4J_URI}")
    print(f"Data Path: {DATA_PATH}")
    print(f"Clear Database: {CLEAR_DATABASE}")

In [ ]:
# Configure Neo4j Spark Connector
spark.conf.set("neo4j.url", NEO4J_URI)
spark.conf.set("neo4j.authentication.basic.username", NEO4J_USERNAME)
spark.conf.set("neo4j.authentication.basic.password", NEO4J_PASSWORD)
spark.conf.set("neo4j.database", "neo4j")

print("Spark configured for Neo4j connection")

## Section 2: The Data Model

The data represents an Aircraft Digital Twin. The graph has nine node types and twelve relationship types.

**Nodes:** Aircraft, System, Component, Sensor, Airport, Flight, Delay, MaintenanceEvent, Removal

**Core topology:**
```
(Aircraft) -[:HAS_SYSTEM]-> (System) -[:HAS_COMPONENT]-> (Component)
```

On top of that topology, Systems carry Sensors, Aircraft operate Flights between Airports (which may have Delays), and Components accumulate Maintenance Events and Removals. We load the core topology first because it is the easiest to reason about, then layer on the operational data.

## Section 3: Helper Functions

These small helpers wrap the Neo4j Spark Connector so the loading cells below stay readable.

- `write_nodes` writes a DataFrame as nodes using plain CREATE, split across a few partitions so batches load in parallel transactions.
- `write_relationships` writes relationships using the `keys` save strategy, matching source and target nodes by their key properties.
- `run_cypher` / `run_script` run read queries and DDL (constraints) respectively.

In [ ]:
from pyspark.sql.functions import col

# Increase from default 5000 — the docs note the default is
# "likely too low for many applications" and recommend ~20,000.
BATCH_SIZE = 20000

# Node writes are safe to parallelize: every row has a distinct key,
# so concurrent transactions never lock the same node.
NODE_PARTITIONS = 4


def read_csv(filename):
    """Read a CSV file from the Unity Catalog Volume."""
    return spark.read.option("header", "true").csv(f"{DATA_PATH}/{filename}")


def write_nodes(df, label):
    """Write a DataFrame as nodes to Neo4j.

    Uses Append mode (plain CREATE) because the database is cleared
    before loading, so per-row MERGE lookups buy nothing. The uniqueness
    constraints from Section 4 guard against accidental duplicates.
    """
    (df.repartition(NODE_PARTITIONS)
     .write
     .format("org.neo4j.spark.DataSource")
     .mode("Append")
     .option("labels", f":{label}")
     .option("batch.size", BATCH_SIZE)
     .save())
    print(f"  [OK] Wrote {label} nodes")


def write_relationships(df, rel_type, source_label, source_key, target_label, target_key):
    """Write relationships to Neo4j using keys strategy.

    Uses coalesce(1) to send all rows through a single partition,
    preventing deadlocks from concurrent writes locking the same nodes.
    """
    (df.coalesce(1)
     .write
     .format("org.neo4j.spark.DataSource")
     .mode("Overwrite")
     .option("relationship", rel_type)
     .option("relationship.save.strategy", "keys")
     .option("relationship.source.labels", f":{source_label}")
     .option("relationship.source.node.keys", source_key)
     .option("relationship.target.labels", f":{target_label}")
     .option("relationship.target.node.keys", target_key)
     .option("batch.size", BATCH_SIZE)
     .save())
    print(f"  [OK] Wrote {rel_type} relationships")


def run_cypher(query):
    """Execute a Cypher query and return results as DataFrame."""
    return (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("query", query)
        .load())


def run_script(script):
    """Execute a Cypher script (DDL operations like constraints).

    Spark reads are lazy: .load() only builds the read, so the script
    would never reach Neo4j without the .collect() forcing execution.
    """
    (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("script", script)
        .option("query", "RETURN 1 AS done")
        .load()
        .collect())


print("Helper functions defined.")

## Section 4: Prepare the Database

Optionally clear any existing data, then create uniqueness constraints and indexes. Constraints matter for loading: the relationship `keys` strategy looks up source and target nodes by their key property, and a uniqueness constraint backs that lookup with an index so it stays fast.

In [ ]:
if CLEAR_DATABASE:
    print("Clearing database...")
    run_script("CALL { MATCH (n) WITH n LIMIT 10000 DETACH DELETE n } IN TRANSACTIONS OF 10000 ROWS")
    result = run_cypher("MATCH (n) RETURN count(n) AS remaining")
    display(result)
    remaining = result.collect()[0]["remaining"]
    if remaining > 0:
        print(f"WARNING: {remaining} nodes still remain. Re-run this cell until remaining = 0.")
    else:
        print("[OK] Database cleared.")
else:
    print("Skipping database clear (CLEAR_DATABASE = False)")

In [ ]:
print("Creating constraints...")

constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Aircraft) REQUIRE n.aircraft_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:System) REQUIRE n.system_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Component) REQUIRE n.component_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Sensor) REQUIRE n.sensor_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Airport) REQUIRE n.airport_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Flight) REQUIRE n.flight_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Delay) REQUIRE n.delay_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:MaintenanceEvent) REQUIRE n.event_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Removal) REQUIRE n.removal_id IS UNIQUE",
]

indexes = [
    "CREATE INDEX idx_maintenanceevent_severity IF NOT EXISTS FOR (n:MaintenanceEvent) ON (n.severity)",
    "CREATE INDEX idx_flight_aircraft_id IF NOT EXISTS FOR (n:Flight) ON (n.aircraft_id)",
    "CREATE INDEX idx_removal_aircraft_id IF NOT EXISTS FOR (n:Removal) ON (n.aircraft_id)",
]

run_script(";\n".join(constraints + indexes))
print("[OK] Constraints and indexes created.")

## Section 5: Load Nodes

Each DataFrame row becomes a node, column values become node properties, and the `labels` option sets the node label (e.g., `:Aircraft`). Nodes are written with plain CREATE in parallel batches; the uniqueness constraints from Section 4 catch any accidental duplicates.

The CSV files use Neo4j import headers like `:ID(Aircraft)`, so for each file we rename that column to a clean property name (e.g., `aircraft_id`) that the constraints and relationship loads rely on.

### Core topology: Aircraft, System, Component

Start with the three node types that form the `(Aircraft)-[:HAS_SYSTEM]->(System)-[:HAS_COMPONENT]->(Component)` backbone.

In [ ]:
print("Loading Aircraft nodes...")
df = read_csv("nodes_aircraft.csv").withColumnRenamed(":ID(Aircraft)", "aircraft_id")

# Inspect the transformed schema so you can see the cleaned-up property names
df.printSchema()

write_nodes(df, "Aircraft")

In [ ]:
print("Loading System nodes...")
df = read_csv("nodes_systems.csv").withColumnRenamed(":ID(System)", "system_id")
write_nodes(df, "System")

In [ ]:
print("Loading Component nodes...")
df = read_csv("nodes_components.csv").withColumnRenamed(":ID(Component)", "component_id")
write_nodes(df, "Component")

### Operational data: Sensors, Airports, Flights, Delays, Maintenance, Removals

These node types hang off the core topology. A few of them need explicit type casts (for example, airport coordinates to `double` and delay minutes to `integer`) because everything arrives from CSV as strings.

In [ ]:
print("Loading Sensor nodes...")
df = read_csv("nodes_sensors.csv").withColumnRenamed(":ID(Sensor)", "sensor_id")
write_nodes(df, "Sensor")

In [ ]:
print("Loading Airport nodes...")
df = (read_csv("nodes_airports.csv")
    .withColumnRenamed(":ID(Airport)", "airport_id")
    .withColumn("lat", col("lat").cast("double"))
    .withColumn("lon", col("lon").cast("double")))
write_nodes(df, "Airport")

In [ ]:
print("Loading Flight nodes...")
df = read_csv("nodes_flights.csv").withColumnRenamed(":ID(Flight)", "flight_id")
write_nodes(df, "Flight")

In [ ]:
print("Loading Delay nodes...")
df = (read_csv("nodes_delays.csv")
    .withColumnRenamed(":ID(Delay)", "delay_id")
    .withColumn("minutes", col("minutes").cast("integer")))
write_nodes(df, "Delay")

In [ ]:
print("Loading MaintenanceEvent nodes...")
df = read_csv("nodes_maintenance.csv").withColumnRenamed(":ID(MaintenanceEvent)", "event_id")
write_nodes(df, "MaintenanceEvent")

In [ ]:
print("Loading Removal nodes...")
df = (read_csv("nodes_removals.csv")
    .withColumnRenamed(":ID(RemovalEvent)", "removal_id")
    .withColumnRenamed("RMV_REA_TX", "reason")
    .withColumnRenamed("time_since_install", "tsn")
    .withColumnRenamed("flight_cycles_at_removal", "csn")
    .withColumn("tsn", col("tsn").cast("double"))
    .withColumn("csn", col("csn").cast("integer")))
write_nodes(df, "Removal")

## Section 6: Load Relationships

Relationships connect the nodes we just loaded. The `keys` save strategy matches existing nodes by their key properties:

- The `relationship` option specifies the relationship type
- `relationship.save.strategy = keys` matches nodes by their key properties
- Source and target labels/keys identify which nodes to connect

As with the nodes, we rename the `:START_ID(...)` and `:END_ID(...)` import columns to the clean key names the strategy expects.

### Core topology: HAS_SYSTEM and HAS_COMPONENT

Connect each Aircraft to its Systems, and each System to its Components.

In [ ]:
print("Loading HAS_SYSTEM relationships...")
df = (read_csv("rels_aircraft_system.csv")
    .withColumnRenamed(":START_ID(Aircraft)", "aircraft_id")
    .withColumnRenamed(":END_ID(System)", "system_id"))
write_relationships(df, "HAS_SYSTEM", "Aircraft", "aircraft_id", "System", "system_id")

In [ ]:
print("Loading HAS_COMPONENT relationships...")
df = (read_csv("rels_system_component.csv")
    .withColumnRenamed(":START_ID(System)", "system_id")
    .withColumnRenamed(":END_ID(Component)", "component_id"))
write_relationships(df, "HAS_COMPONENT", "System", "system_id", "Component", "component_id")

### Operational relationships

Connect the remaining node types: Sensors to Systems, Maintenance Events and Removals to Components and Aircraft, and Flights to Aircraft, Airports, and Delays.

In [ ]:
print("Loading HAS_SENSOR relationships...")
df = (read_csv("rels_system_sensor.csv")
    .withColumnRenamed(":START_ID(System)", "system_id")
    .withColumnRenamed(":END_ID(Sensor)", "sensor_id"))
write_relationships(df, "HAS_SENSOR", "System", "system_id", "Sensor", "sensor_id")

In [ ]:
print("Loading HAS_EVENT relationships...")
df = (read_csv("rels_component_event.csv")
    .withColumnRenamed(":START_ID(Component)", "component_id")
    .withColumnRenamed(":END_ID(MaintenanceEvent)", "event_id"))
write_relationships(df, "HAS_EVENT", "Component", "component_id", "MaintenanceEvent", "event_id")

In [ ]:
print("Loading OPERATES_FLIGHT relationships...")
df = (read_csv("rels_aircraft_flight.csv")
    .withColumnRenamed(":START_ID(Aircraft)", "aircraft_id")
    .withColumnRenamed(":END_ID(Flight)", "flight_id"))
write_relationships(df, "OPERATES_FLIGHT", "Aircraft", "aircraft_id", "Flight", "flight_id")

In [ ]:
print("Loading DEPARTS_FROM relationships...")
df = (read_csv("rels_flight_departure.csv")
    .withColumnRenamed(":START_ID(Flight)", "flight_id")
    .withColumnRenamed(":END_ID(Airport)", "airport_id"))
write_relationships(df, "DEPARTS_FROM", "Flight", "flight_id", "Airport", "airport_id")

In [ ]:
print("Loading ARRIVES_AT relationships...")
df = (read_csv("rels_flight_arrival.csv")
    .withColumnRenamed(":START_ID(Flight)", "flight_id")
    .withColumnRenamed(":END_ID(Airport)", "airport_id"))
write_relationships(df, "ARRIVES_AT", "Flight", "flight_id", "Airport", "airport_id")

In [ ]:
print("Loading HAS_DELAY relationships...")
df = (read_csv("rels_flight_delay.csv")
    .withColumnRenamed(":START_ID(Flight)", "flight_id")
    .withColumnRenamed(":END_ID(Delay)", "delay_id"))
write_relationships(df, "HAS_DELAY", "Flight", "flight_id", "Delay", "delay_id")

In [ ]:
print("Loading AFFECTS_SYSTEM relationships...")
df = (read_csv("rels_event_system.csv")
    .withColumnRenamed(":START_ID(MaintenanceEvent)", "event_id")
    .withColumnRenamed(":END_ID(System)", "system_id"))
write_relationships(df, "AFFECTS_SYSTEM", "MaintenanceEvent", "event_id", "System", "system_id")

In [ ]:
print("Loading AFFECTS_AIRCRAFT relationships...")
df = (read_csv("rels_event_aircraft.csv")
    .withColumnRenamed(":START_ID(MaintenanceEvent)", "event_id")
    .withColumnRenamed(":END_ID(Aircraft)", "aircraft_id"))
write_relationships(df, "AFFECTS_AIRCRAFT", "MaintenanceEvent", "event_id", "Aircraft", "aircraft_id")

In [ ]:
print("Loading HAS_REMOVAL relationships...")
df = (read_csv("rels_aircraft_removal.csv")
    .withColumnRenamed(":START_ID(Aircraft)", "aircraft_id")
    .withColumnRenamed(":END_ID(RemovalEvent)", "removal_id"))
write_relationships(df, "HAS_REMOVAL", "Aircraft", "aircraft_id", "Removal", "removal_id")

In [ ]:
print("Loading REMOVED_COMPONENT relationships...")
df = (read_csv("rels_component_removal.csv")
    .withColumnRenamed(":START_ID(Component)", "component_id")
    .withColumnRenamed(":END_ID(RemovalEvent)", "removal_id"))
write_relationships(df, "REMOVED_COMPONENT", "Removal", "removal_id", "Component", "component_id")

## Section 7: Verification Queries

Verify the data loaded correctly by running Cypher queries from Databricks.

In [ ]:
print("Node counts by label:")
result = run_cypher("""
    CALL () {
        MATCH (n:Aircraft) RETURN 'Aircraft' as label, count(n) as count
        UNION ALL
        MATCH (n:System) RETURN 'System' as label, count(n) as count
        UNION ALL
        MATCH (n:Component) RETURN 'Component' as label, count(n) as count
        UNION ALL
        MATCH (n:Sensor) RETURN 'Sensor' as label, count(n) as count
        UNION ALL
        MATCH (n:Airport) RETURN 'Airport' as label, count(n) as count
        UNION ALL
        MATCH (n:Flight) RETURN 'Flight' as label, count(n) as count
        UNION ALL
        MATCH (n:Delay) RETURN 'Delay' as label, count(n) as count
        UNION ALL
        MATCH (n:MaintenanceEvent) RETURN 'MaintenanceEvent' as label, count(n) as count
        UNION ALL
        MATCH (n:Removal) RETURN 'Removal' as label, count(n) as count
    }
    RETURN label, count
    ORDER BY count DESC
""")
display(result)

In [ ]:
print("Relationship counts by type:")
result = run_cypher("""
    MATCH ()-[r]->()
    RETURN type(r) AS RelType, count(*) AS Count
    ORDER BY Count DESC
""")
display(result)

## ETL Complete!

Your Neo4j database now contains the complete Aircraft Digital Twin graph: Aircraft, Systems, Components, Sensors, Airports, Flights, Delays, Maintenance Events, and Removals, with all of their relationships. This is the graph that Labs 3 and 4 build on.

### Explore in Neo4j Aura

1. Go to [console.neo4j.io](https://console.neo4j.io)
2. Open your instance and click **Query**

**See one aircraft's complete hierarchy:**
```cypher
MATCH (a:Aircraft {tail_number: 'N10000'})-[r1:HAS_SYSTEM]->(s:System)-[r2:HAS_COMPONENT]->(c:Component)
RETURN a, r1, s, r2, c
```

**View the complete schema:**
```cypher
CALL db.schema.visualization()
```

**Find aircraft with critical maintenance issues:**
```cypher
MATCH (a:Aircraft)-[:HAS_SYSTEM]->(s:System)-[:HAS_COMPONENT]->(c:Component)-[:HAS_EVENT]->(m:MaintenanceEvent)
WHERE m.severity = 'CRITICAL' AND m.reported_at IS NOT NULL
RETURN a.tail_number, s.name, c.name, m.fault, m.reported_at
ORDER BY m.reported_at DESC
LIMIT 10
```

**Analyze flight delays by cause:**
```cypher
MATCH (f:Flight)-[:HAS_DELAY]->(d:Delay)
RETURN d.cause, count(*) AS count, avg(d.minutes) AS avg_minutes
ORDER BY count DESC
```

### Next Steps

Proceed to **Lab 3 - Semantic Search** to load the maintenance manuals, generate vector embeddings, and build GraphRAG retrievers on top of this graph.